<a href="https://colab.research.google.com/github/Nipun1a/fastAPI_practice/blob/main/fastapi_insurance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [82]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split # Added this import
import numpy as np
import pandas as pd

In [83]:
df = pd.read_csv('/content/insurance.csv')

In [84]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
39,51,100.6,1.68,11.99,True,Bangalore,unemployed,High
18,52,80.9,1.80,38.14,True,Kota,business_owner,High
25,59,60.2,1.55,30.00,False,Mysore,government_job,Low
84,75,86.2,1.73,0.62,True,Jaipur,retired,High
66,18,63.9,1.59,3.23,False,Indore,student,Low


In [85]:
df_feat = df.copy()

In [86]:
#feature 1: BMI
df_feat["bmi"] = df_feat["weight"] / (df_feat["height"] ** 2)

In [87]:
#Feature 2: Age group
def age_group(age):
  if age < 25:
    return "young"
  elif age < 45:
    return "adult"
  elif age < 60:
    return "middle_aged"
  return "senior"

In [88]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [89]:
# feature 3: Lifestyle Risk
def lifestyle_risk(row):
  if row["smoker"] and row["bmi"] > 30:
    return "high"
  elif row["smoker"] or row["bmi"] > 27:
    return "medium"
  else:
    return "low"

In [90]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [91]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
"Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
"Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
"Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
"Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
"Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
"Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

# Feature 4: City Tier
def city_tiers(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2
  else:
    return 3

In [92]:
# Feature 4: City Tier
def city_tiers(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2
  else:
    return 3

In [93]:
df_feat["city_tiers"] = df_feat["city"].apply(city_tiers)

In [94]:
df_feat.drop(columns=["age","weight","height","smoker","city"])[['income_lpa','occupation',"bmi",'age_group','lifestyle_risk','city_tiers','insurance_premium_category']]

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tiers,insurance_premium_category
0,2.92000,retired,49.227482,senior,medium,2,High
1,34.28000,freelancer,30.189017,adult,medium,1,Low
2,36.64000,freelancer,21.118382,adult,low,2,Low
3,3.34000,student,45.535900,young,high,1,Medium
4,3.94000,retired,24.296875,senior,medium,2,High
...,...,...,...,...,...,...,...
95,19.64000,business_owner,21.420747,adult,low,2,Low
96,34.01000,private_job,47.984483,adult,medium,1,Low
97,44.86000,freelancer,18.765432,middle_aged,low,1,Low
98,28.30000,business_owner,30.521676,adult,medium,1,Low


In [95]:
#select features and target
x= df_feat[["bmi","age_group","lifestyle_risk","city_tiers","income_lpa","occupation"]]
y= df_feat["insurance_premium_category"]


In [96]:
x


,bmi,age_group,lifestyle_risk,city_tiers,income_lpa,occupation
0,49.227482,senior,medium,2,2.92000,retired
1,30.189017,adult,medium,1,34.28000,freelancer
2,21.118382,adult,low,2,36.64000,freelancer
3,45.535900,young,high,1,3.34000,student
4,24.296875,senior,medium,2,3.94000,retired
...,...,...,...,...,...,...
95,21.420747,adult,low,2,19.64000,business_owner
96,47.984483,adult,medium,1,34.01000,private_job
97,18.765432,middle_aged,low,1,44.86000,freelancer
98,30.521676,adult,medium,1,28.30000,business_owner


In [97]:
#define categorical and numeric features
categorical_features = ["age_group","lifestyle_risk","city_tiers","occupation"]
numeric_features = ["bmi","income_lpa"]

In [98]:
#create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [99]:
#create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))

])

In [101]:
#Split data and train model
X_train,X_test,y_train,y_test = train_test_split(x,y, test_size=0.2,random_state = 1)
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'city_tiers',
                                                   'occupation']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [102]:
#Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test,y_pred)

0.8

In [104]:
X_test.sample(5)

,bmi,age_group,lifestyle_risk,city_tiers,income_lpa,occupation
69,21.942857,middle_aged,low,2,6.034487,government_job
78,27.932798,middle_aged,medium,2,14.740000,freelancer
44,30.078125,middle_aged,high,2,50.000000,private_job
10,22.949982,adult,medium,1,32.780000,business_owner
84,28.801497,senior,medium,2,0.620000,retired


In [105]:
import pickle
#save the trained pipeline using pickle

pickle_model_path = 'model.pkl'
with open(pickle_model_path, 'wb') as file:
    pickle.dump(pipeline, file)